# Apilytics Quickstart (Scala)

Query REST APIs with Spark SQL using Apilytics.

## Setup

This notebook creates a SparkSession with two catalogs configured:
- `pokeapi` - Query PokeAPI (pokemon, types, abilities)
- `github` - Query GitHub API (issues from octocat/Hello-World)

**Note:** This notebook requires the Scala kernel (Almond). Start Jupyter with:
```bash
cd docker/spark
docker compose -f compose.spark.yaml up jupyter
```

In [ ]:
// Add Spark and Apilytics dependencies
import $ivy.`org.apache.spark::spark-sql:4.0.0`
import $cp.`/opt/spark/jars/apilytics.jar`

import org.apache.spark.sql.SparkSession

// Create SparkSession with both PokeAPI and GitHub catalogs
val spark = SparkSession.builder()
  .appName("Apilytics Scala Notebook")
  .master("local[*]")
  .config("spark.sql.catalog.pokeapi", "com.apilytics.spark.RESTCatalog")
  .config("spark.sql.catalog.pokeapi.config", "/opt/apilytics/examples/pokeapi/pokeapi-config.conf")
  .config("spark.sql.catalog.github", "com.apilytics.spark.RESTCatalog")
  .config("spark.sql.catalog.github.config", "/opt/apilytics/examples/github/github-config.conf")
  .getOrCreate()

import spark.implicits._
println(s"Spark version: ${spark.version}")
println("Catalogs configured: pokeapi, github")

## Basic Queries

Query the PokeAPI like a database table:

In [ ]:
// List available tables in PokeAPI
spark.sql("SHOW TABLES IN pokeapi.default").show()

In [ ]:
// Query Pokemon
spark.sql("SELECT name, url FROM pokeapi.default.pokemon LIMIT 10").show(false)

In [ ]:
// Check the schema
spark.sql("DESCRIBE pokeapi.default.pokemon").show()

## DataFrame API

You can also use the DataFrame API:

In [ ]:
// Load as DataFrame
val pokemon = spark.table("pokeapi.default.pokemon")

// Filter and select
pokemon.filter($"name".startsWith("char")).show()

In [ ]:
// Count
println(s"Total Pokemon: ${pokemon.count()}")

## Caching

Each query hits the API. For repeated analysis, cache locally:

In [ ]:
// Cache for faster repeated queries
val pokemonCached = spark.table("pokeapi.default.pokemon").cache()

// Now these queries are fast
pokemonCached.groupBy("name").count().show(5)
pokemonCached.filter($"name".like("%saur")).show()

## Clean Error Messages

Use `sqlClean()` for user-friendly error messages:

In [ ]:
// Query GitHub issues
spark.sql("SELECT number, title, state FROM github.default.issues LIMIT 5").show(false)